# OFT Serving

SGLang supports [Orthogonal Finetuning (OFT)](https://arxiv.org/abs/2306.07280) adapters with a base model. OFT is a parameter-efficient finetuning method that applies learned orthogonal rotations to the model's weight matrices, preserving the hyperspherical energy of pretrained features while adapting the model to new tasks.

Unlike LoRA which adds low-rank additive updates (`output = base_output + x @ A^T @ B^T`), OFT applies a multiplicative orthogonal rotation to the input before the linear layer (`output = (x @ R) @ W^T`), where `R` is a block-diagonal orthogonal matrix parameterized via the Cayley transform of skew-symmetric blocks.

SGLang supports multi-OFT serving analogous to multi-LoRA, efficiently handling different OFT adapters for different sequences within a single batch.

## Arguments for OFT Serving

The following server arguments are relevant for multi-OFT serving:

* `enable_oft`: Enable OFT support for the model. This argument is automatically set to True if `--oft-paths` is provided for backward compatibility.

* `enable_oft_overlap_loading`: Enable asynchronous OFT weight loading in order to overlap H2D transfers with GPU compute. This should be enabled if you find that your OFT workloads are bottlenecked by adapter weight loading, for example when frequently loading large OFT adapters.

* `oft_paths`: The list of OFT adapters to load. Each adapter must be specified in one of the following formats: <PATH> | <NAME>=<PATH> | JSON with schema {"oft_name":str,"oft_path":str,"pinned":bool}.

* `max_ofts_per_batch`: Maximum number of adapters used by each batch. This argument can affect the amount of GPU memory reserved for multi-OFT serving, so it should be set to a smaller value when memory is scarce. Defaults to 8.

* `max_loaded_ofts`: If specified, it limits the maximum number of OFT adapters loaded in CPU memory at a time. The value must be greater than or equal to `max-ofts-per-batch`.

* `oft_eviction_policy`: OFT adapter eviction policy when GPU memory pool is full. `lru`: Least Recently Used (default, better cache efficiency). `fifo`: First-In-First-Out.

* `oft_backend`: The backend for running OFT kernels. Currently only `torch_native` is supported.

* `max_oft_block_size`: The maximum block size of OFT adapters (controls the orthogonal block dimensions). If not specified, it will be automatically inferred from the adapters provided in `--oft-paths`. This argument is needed when you expect to dynamically load adapters of larger block size after server startup.

* `oft_target_modules`: The union set of all target modules where OFT should be applied (e.g., `q_proj`, `k_proj`, `gate_proj`). If not specified, it will be automatically inferred from the adapters provided in `--oft-paths`. This argument is needed when you expect to dynamically load adapters of different target modules after server startup. You can also set it to `all` to enable OFT for all supported modules. However, enabling OFT on additional modules introduces a minor performance overhead.

* `--max-oft-chunk-size`: Maximum chunk size for OFT operations. Must be a power of 2 between 16 and 128. Defaults to 16.

* `tp_size`: OFT serving along with Tensor Parallelism is supported by SGLang. `tp_size` controls the number of GPUs for tensor parallelism.

From client side, the user needs to provide a list of strings as input batch, and a list of adapter names that each input sequence corresponds to.

## Usage

### Serving Single Adapter

**Note:** SGLang supports OFT adapters through two APIs:

1. **OpenAI-Compatible API** (`/v1/chat/completions`, `/v1/completions`): Use the `model:adapter-name` syntax.

2. **Native API** (`/generate`): Pass `oft_path` in the request body (shown below).

In [ ]:
import json
import requests

from sglang.test.doc_patch import launch_server_cmd
from sglang.utils import wait_for_server, terminate_process

In [ ]:
server_process, port = launch_server_cmd(
    # Here we set max-ofts-per-batch to 2: one slot for adapter and another one for base model
    """
python3 -m sglang.launch_server --model-path meta-llama/Meta-Llama-3.1-8B-Instruct \
    --enable-oft \
    --oft-paths oft0=path/to/oft-adapter-0 \
    --max-ofts-per-batch 2 \
    --log-level warning \
"""
)

wait_for_server(f"http://localhost:{port}")

In [ ]:
url = f"http://127.0.0.1:{port}"
json_data = {
    "text": [
        "List 3 countries and their capitals.",
        "List 3 countries and their capitals.",
    ],
    "sampling_params": {"max_new_tokens": 32, "temperature": 0},
    # The first input uses oft0, and the second input uses the base model
    "oft_path": ["oft0", None],
}
response = requests.post(
    url + "/generate",
    json=json_data,
)
print(f"Output 0: {response.json()[0]['text']}")
print(f"Output 1: {response.json()[1]['text']}")

In [ ]:
terminate_process(server_process)

### Serving Multiple Adapters

In [ ]:
server_process, port = launch_server_cmd(
    """
python3 -m sglang.launch_server --model-path meta-llama/Meta-Llama-3.1-8B-Instruct \
    --enable-oft \
    --oft-paths oft0=path/to/oft-adapter-0 \
    oft1=path/to/oft-adapter-1 \
    --max-ofts-per-batch 2 \
    --log-level warning \
"""
)

wait_for_server(f"http://localhost:{port}")

In [ ]:
url = f"http://127.0.0.1:{port}"
json_data = {
    "text": [
        "List 3 countries and their capitals.",
        "List 3 countries and their capitals.",
    ],
    "sampling_params": {"max_new_tokens": 32, "temperature": 0},
    # The first input uses oft0, and the second input uses oft1
    "oft_path": ["oft0", "oft1"],
}
response = requests.post(
    url + "/generate",
    json=json_data,
)
print(f"Output 0: {response.json()[0]['text']}")
print(f"Output 1: {response.json()[1]['text']}")

In [ ]:
terminate_process(server_process)

### Dynamic OFT Loading

Instead of specifying all adapters during server startup via `--oft-paths`, you can also load & unload OFT adapters dynamically via the `/load_oft_adapter` and `/unload_oft_adapter` API.

When using dynamic OFT loading, it's recommended to explicitly specify both `--max-oft-block-size` and `--oft-target-modules` at startup. For backward compatibility, SGLang will infer these values from `--oft-paths` if they are not explicitly provided. However, in that case, you would have to ensure that all dynamically loaded adapters share the same shape (block size and target modules) as those in the initial `--oft-paths` or are strictly "smaller".

In [ ]:
oft0 = "path/to/oft-adapter-0"
oft1 = "path/to/oft-adapter-1"
oft0_new = "path/to/oft-adapter-0-new"


# The `--oft-target-modules` param below is technically not needed if the server can infer it
# from the initial adapters. We are adding it here just to demonstrate usage.
server_process, port = launch_server_cmd(
    """
    python3 -m sglang.launch_server --model-path meta-llama/Meta-Llama-3.1-8B-Instruct \
    --enable-oft \
    --cuda-graph-max-bs 2 \
    --max-ofts-per-batch 2 \
    --max-oft-block-size 256 \
    --oft-target-modules all \
    --log-level warning
    """
)

url = f"http://127.0.0.1:{port}"
wait_for_server(url)

Load adapter oft0

In [ ]:
response = requests.post(
    url + "/load_oft_adapter",
    json={
        "oft_name": "oft0",
        "oft_path": oft0,
    },
)

if response.status_code == 200:
    print("OFT adapter loaded successfully.", response.json())
else:
    print("Failed to load OFT adapter.", response.json())

Load adapter oft1:

In [ ]:
response = requests.post(
    url + "/load_oft_adapter",
    json={
        "oft_name": "oft1",
        "oft_path": oft1,
    },
)

if response.status_code == 200:
    print("OFT adapter loaded successfully.", response.json())
else:
    print("Failed to load OFT adapter.", response.json())

Check inference output:

In [ ]:
url = f"http://127.0.0.1:{port}"
json_data = {
    "text": [
        "List 3 countries and their capitals.",
        "List 3 countries and their capitals.",
    ],
    "sampling_params": {"max_new_tokens": 32, "temperature": 0},
    # The first input uses oft0, and the second input uses oft1
    "oft_path": ["oft0", "oft1"],
}
response = requests.post(
    url + "/generate",
    json=json_data,
)
print(f"Output from oft0: \n{response.json()[0]['text']}\n")
print(f"Output from oft1 (updated): \n{response.json()[1]['text']}\n")

Unload oft0 and replace it with a different adapter:

In [ ]:
response = requests.post(
    url + "/unload_oft_adapter",
    json={
        "oft_name": "oft0",
    },
)

response = requests.post(
    url + "/load_oft_adapter",
    json={
        "oft_name": "oft0",
        "oft_path": oft0_new,
    },
)

if response.status_code == 200:
    print("OFT adapter loaded successfully.", response.json())
else:
    print("Failed to load OFT adapter.", response.json())

Check output again:

In [ ]:
url = f"http://127.0.0.1:{port}"
json_data = {
    "text": [
        "List 3 countries and their capitals.",
        "List 3 countries and their capitals.",
    ],
    "sampling_params": {"max_new_tokens": 32, "temperature": 0},
    # The first input uses oft0, and the second input uses oft1
    "oft_path": ["oft0", "oft1"],
}
response = requests.post(
    url + "/generate",
    json=json_data,
)
print(f"Output from oft0: \n{response.json()[0]['text']}\n")
print(f"Output from oft1 (updated): \n{response.json()[1]['text']}\n")

### OpenAI-compatible API usage

You can use OFT adapters via the OpenAI-compatible APIs by specifying the adapter in the `model` field using the `base-model:adapter-name` syntax (for example, `meta-llama/Meta-Llama-3.1-8B-Instruct:oft0`). For more details and examples, see the "Using LoRA Adapters" section in the OpenAI API documentation: [openai_api_completions.ipynb](../basic_usage/openai_api_completions.ipynb).

In [ ]:
terminate_process(server_process)

### OFT GPU Pinning

Another advanced option is to specify adapters as `pinned` during loading. When an adapter is pinned, it is permanently assigned to one of the available GPU pool slots (as configured by `--max-ofts-per-batch`) and will not be evicted from GPU memory during runtime. Instead, it remains resident until it is explicitly unloaded.

This can improve performance in scenarios where the same adapter is frequently used across requests, by avoiding repeated memory transfers and reinitialization overhead. However, since GPU pool slots are limited, pinning adapters reduces the flexibility of the system to dynamically load other adapters on demand. If too many adapters are pinned, it may lead to degraded performance, or in the most extreme case (`Number of pinned adapters == max-ofts-per-batch`), halt all unpinned requests. Therefore, currently SGLang limits maximal number of pinned adapters to `max-ofts-per-batch - 1` to prevent unexpected starvations.

In the example below, we start a server with `oft0` loaded as pinned, `oft1` and `oft2` loaded as regular (unpinned) adapters. Please note that, we intentionally specify `oft1` and `oft2` in two different formats to demonstrate that both are supported.

In [ ]:
server_process, port = launch_server_cmd(
    """
    python3 -m sglang.launch_server --model-path meta-llama/Meta-Llama-3.1-8B-Instruct \
    --enable-oft \
    --cuda-graph-max-bs 8 \
    --max-ofts-per-batch 3 \
    --max-oft-block-size 256 \
    --oft-target-modules all \
    --oft-paths \
        {"oft_name":"oft0","oft_path":"path/to/oft-adapter-0","pinned":true} \
        {"oft_name":"oft1","oft_path":"path/to/oft-adapter-1"} \
        oft2=path/to/oft-adapter-2
    --log-level warning
    """
)


url = f"http://127.0.0.1:{port}"
wait_for_server(url)

You can also specify adapter as pinned during dynamic adapter loading. In the example below, we reload `oft1` as pinned adapter:

In [ ]:
response = requests.post(
    url + "/unload_oft_adapter",
    json={
        "oft_name": "oft1",
    },
)

response = requests.post(
    url + "/load_oft_adapter",
    json={
        "oft_name": "oft1",
        "oft_path": "path/to/oft-adapter-1",
        "pinned": True,  # Pin the adapter to GPU
    },
)

Verify that the results are expected:

In [ ]:
url = f"http://127.0.0.1:{port}"
json_data = {
    "text": [
        "List 3 countries and their capitals.",
        "List 3 countries and their capitals.",
        "List 3 countries and their capitals.",
    ],
    "sampling_params": {"max_new_tokens": 32, "temperature": 0},
    # The first input uses oft0, and the second input uses oft1
    "oft_path": ["oft0", "oft1", "oft2"],
}
response = requests.post(
    url + "/generate",
    json=json_data,
)
print(f"Output from oft0 (pinned): \n{response.json()[0]['text']}\n")
print(f"Output from oft1 (pinned): \n{response.json()[1]['text']}\n")
print(f"Output from oft2 (not pinned): \n{response.json()[2]['text']}\n")

In [ ]:
terminate_process(server_process)

## Choosing OFT Backend

SGLang currently supports one OFT backend that you can select using the `--oft-backend` argument:

- `torch_native`: Default PyTorch-based backend.

Additional optimized backends (e.g., Triton, Chunked SGMV) are under development.

In [ ]:
server_process, port = launch_server_cmd(
    """
    python3 -m sglang.launch_server \
    --model-path meta-llama/Meta-Llama-3.1-8B-Instruct \
    --enable-oft \
    --oft-backend torch_native \
    --max-ofts-per-batch 16 \
    --oft-paths oft0=path/to/oft0 oft1=path/to/oft1
    """
)

In [ ]:
terminate_process(server_process)

## Future Works

The OFT implementation in SGLang is actively evolving. Planned improvements include optimized Triton and Chunked SGMV backends for higher throughput, as well as broader integration with additional model architectures.